# DistilBERT Fine-tuning with LoRA for SMS Scam Detection

This notebook fine-tunes DistilBERT using LoRA (Low-Rank Adaptation) for efficient training on SMS scam detection.

##  Install Required Libraries

In [ ]:
# Install transformers, peft (for LoRA), and other dependencies
!pip install -q transformers datasets accelerate peft torch evaluate scikit-learn imbalanced-learn matplotlib seaborn plotly

print(" libraries installed successfully!")

##  Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
import torch
import transformers
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    DistilBertModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    AutoTokenizer, AutoModelForCausalLM,AutoModelForSequenceClassification,AutoModel
)
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    roc_auc_score
)
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
import seaborn as sns
import warnings
from tqdm import tqdm

import os

# Suppress warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("Libraries imported successfully!")

##  Load the Prepared Data

Tis bock will load the trained and test data sets `train_data.csv` and `test_data.csv`

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("Google Drive mounted successfully!")



In [ ]:
# Load the datasets
print("Loading training and test data...")
model_output_dir='/content/drive/MyDrive/scam_detection/distilbert_lora_sms_scam/'
os.makedirs(model_output_dir, exist_ok=True)

parent_dir="/content/drive/MyDrive/scam_detection/Qwen_scam_datasets_processed"

try:
  train_df = pd.read_csv(f'{parent_dir}/train_scam_dataset.csv')
  test_df = pd.read_csv(f'{parent_dir}/test_scam_dataset.csv')


  print(f"Training data loaded: {len(train_df)} samples")
  print(f"Test data loaded: {len(test_df)} samples")

  # Display class distribution
  print(f"\nTraining set distribution:")
  print(f"Not Scam: {len(train_df[train_df['label'] == 0])} ({len(train_df[train_df['label'] == 0])/len(train_df)*100:.1f}%)")
  print(f"Scam: {len(train_df[train_df['label'] == 1])} ({len(train_df[train_df['label'] == 1])/len(train_df)*100:.1f}%)")

  train_df.head()
except Exception as e:
  print(f"Error loading data: {e}")


##  Load Tokenizer and Model

In [ ]:
# Load tokenizer
print("Loading distilbert tokenizer...")

model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
embedding_model = DistilBertModel.from_pretrained(model_name)
embedding_model.to(device)
embedding_model.eval()

print(" distilbert model loaded successfully!")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "not_scam", 1: "scam"},
    label2id={"not_scam": 0, "scam": 1}
).to(device)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded successfully!")
print(f"\nModel size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

##  Prepare Data for Qwen



In [ ]:
# Convert to Hugging Face datasets
print("Converting to Hugging Face Dataset format...")

train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("Dataset conversion complete!")
print(dataset_dict)

##  Tokenize the Data

In [ ]:
# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

print("Tokenizing datasets...")
tokenized_datasets = dataset_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print("Tokenization complete!")
print(tokenized_datasets)

##  Configure LoRA (Low-Rank Adaptation)

LoRA allows us to fine-tune the model efficiently by only training a small number of additional parameters

In [ ]:
# Configure LoRA
print("Configuring LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence Classification
    r=32,  # Rank of the low-rank matrices (higher = more capacity, but more parameters)
    lora_alpha=64,  # Scaling factor
    lora_dropout=0.1,  # Dropout probability
    target_modules=["q_lin", "v_lin"],  # Which layers to apply LoRA to
    bias="none",
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\n LoRA configured successfully!")

## Define Training Arguments

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=model_output_dir,
    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=100,

    bf16=True,
    dataloader_num_workers=2,

    report_to="none",
)

print("Training arguments configured!")

## Define Evaluation Metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Metrics function defined!")



## Initialize Trainer and Start Fine-tuning

In [ ]:

#since dataset has more of not scam than scam , we will implement class weights to ensure the model focuses more on learning scam patterns than not scam

labels = train_df['label'].values

print(labels[:10])

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Use weighted cross entropy loss
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
# Initialize Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
)

print("Trainer initialized!")
print("\n" + "="*80)
print("STARTING FINE-TUNING...")
print("="*80)

# Start training
train_result = trainer.train()

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)

##  Save the Fine-tuned Model

In [ ]:
# Save the model and tokenizer
print("Saving fine-tuned model...")

trainer.save_model(f"{model_output_dir}distilbert_lora_sms_scam_final")
tokenizer.save_pretrained(f"{model_output_dir}distilbert_lora_sms_scam_final")

print("Model saved successfully!")

##  Evaluate on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
print("="*80)

eval_results = trainer.evaluate()

print("\n TEST SET RESULTS:")
print("="*80)
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

print("="*80)

## Make Predictions on Test Set

In [ ]:
# Get predictions
print("Getting predictions on test set...")

predictions = trainer.predict(tokenized_datasets['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("Predictions complete!")

## Detailed Performance Analysis

In [ ]:
# Calculate detailed metrics
from sklearn.metrics import classification_report

print("\n DETAILED CLASSIFICATION REPORT:")
print("="*80)
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['Not Scam', 'Scam'],
    digits=4
))
print("="*80)

##  Confusion Matrix Visualization

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Scam', 'Scam'],
    yticklabels=['Not Scam', 'Scam']
)
plt.title('Confusion Matrix - SMS Scam Detection', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved as 'confusion_matrix.png'")

##  Test with Custom Messages

In [ ]:
# Function to predict custom messages
def predict_message(text, model, tokenizer):
    """
    Predict if a message is scam or not scam
    """
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()

    label = "SCAM" if predicted_class == 1 else "NOT SCAM"
    return label, confidence

# Move model to device
model.to(device)

print("Prediction function ready!")

In [ ]:
# Test with sample messages
test_messages = [
    "Hey, are we still meeting for dinner tonight?",
    "CONGRATULATIONS! You've won a FREE iPhone! Click here to claim now!",
    "Can you pick up some milk on your way home?",
    "URGENT: Your bank account has been compromised. Call us immediately at 555-0000",
    "Happy birthday! Hope you have an amazing day!",
    "Win cash prizes! Text WIN to 12345 now! Offer expires soon!"
]

print("\n TESTING CUSTOM MESSAGES:")
print("="*80)

for i, message in enumerate(test_messages, 1):
    label, confidence = predict_message(message, model, tokenizer)
    print(f"\n{i}. Message: {message}")
    print(f"   Prediction: {label} (Confidence: {confidence*100:.2f}%)")
    print("-"*80)

print("="*80)

## Interactive Testing

test the model performance with your own message

In [ ]:
message = input(" Enter testing message ")

label, confidence = predict_message(message, model, tokenizer)

print("\n" + "="*80)
print("YOUR MESSAGE PREDICTION:")
print("="*80
)
print(f"\nPrediction: {label}")
print(f"Confidence: {confidence*100:.2f}%")
print("="*80)

##Analyze Misclassifications

In [ ]:
# Find misclassified examples
test_texts = test_df['text'].values
misclassified_indices = np.where(pred_labels != true_labels)[0]

print(f"\nMISCLASSIFICATIONS: {len(misclassified_indices)} out of {len(test_texts)}")
print(f"Error Rate: {len(misclassified_indices)/len(test_texts)*100:.2f}%")
print("="*80)

# Show first 5 misclassifications
print("\nSample Misclassifications:")
for i, idx in enumerate(misclassified_indices[:5], 1):
    true_label = "SCAM" if true_labels[idx] == 1 else "NOT SCAM"
    pred_label = "SCAM" if pred_labels[idx] == 1 else "NOT SCAM"
    print(f"\n{i}. Text: {test_texts[idx]}")
    print(f"   True Label: {true_label}")
    print(f"   Predicted: {pred_label}")
    print("-"*80)

## Final step: Save Predictions to Drive





In [ ]:
results_df = test_df.copy()
results_df['predicted_label'] = pred_labels
results_df['true_label'] = true_labels
results_df['correct'] = results_df['predicted_label'] == results_df['true_label']

results_df.to_csv('test_predictions.csv', index=False)

print("Predictions saved to 'test_predictions.csv'")
print(f"\n Accuracy: {results_df['correct'].sum() / len(results_df) * 100:.2f}%")